In [ ]:
# feedback of experiment using csv data (pandas)

import pandas as pd

path = '/home/loai/Documents/code/RSMLExtraction/Results/Reconstruction/results_per_box_evaluator.csv'
df = pd.read_csv(path)
df

In [ ]:
# get all unique models 
unique_models = df['model'].unique()
print("Unique models:", unique_models)

unique_boxes = df['box'].unique()
print("Unique boxes:", unique_boxes)

unique_metrics = df['metric'].unique()
print("Unique metrics:", unique_metrics)

In [ ]:
import matplotlib.pyplot as plt

# get one box 
box = unique_boxes[0]
df_box = df[df['box'] == box]

# for first model
model = unique_models[0]
df_model = df_box[df_box['model'] == model]

# plot the first metric for this model
for metric in unique_metrics:
    df_metric = df_model[df_model['metric'] == metric]

    df_metric_expertized = df_metric[df_metric['status'] == 'expertized']
    df_metric_bexpertized = df_metric[df_metric['status'] == 'before_expertized']

    plt.figure()
    plt.plot(df_metric_expertized['time'], df_metric_expertized['value'])
    plt.plot(df_metric_bexpertized['time'], df_metric_bexpertized['value'], linestyle='--')
    plt.title(f'Box: {box}, Model: {model}, Metric: {metric}')
    plt.legend(['Expertized', 'Before Expertized'])
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 0. Filtrer sur split = 'Val'
df_val = df[df['split'] == 'Val']

# 1. Calculer la moyenne et l'écart‑type across boxes
df_stats = (
    df_val
    .groupby(['model', 'metric', 'status', 'time'], as_index=False)['value']
    .agg(mean='mean', std='std')
)

# 2. Boucler sur chaque metric pour tracer mean ± std
for metric in unique_metrics:
    fig, (ax_e, ax_b) = plt.subplots(
        nrows=1, ncols=2, figsize=(14, 6), sharey=True
    )

    df_m = df_stats[df_stats['metric'] == metric]

    for model in unique_models:
        # Extraire mean et std pour chaque statut
        for ax, status, title, linestyle in [
            (ax_e, 'expertized', 'Expertized', '-'),
            (ax_b, 'before_expertized', 'Before Expertized', '--')
        ]:
            df_ms = df_m[
                (df_m['model'] == model) &
                (df_m['status'] == status)
                ].sort_values('time')

            # Séries
            times = df_ms['time']
            means = df_ms['mean']
            stds = df_ms['std'].fillna(0)  # éviter NaN si une seule box

            # Tracer la moyenne
            ax.plot(times, means, label=model, linestyle=linestyle)
            # Bande d'écart‑type
            ax.fill_between(
                times,
                means - stds,
                means + stds,
                alpha=0.2
            )

    # Mise en forme
    ax_e.set_title('Expertized (moyenne ± std)')
    ax_b.set_title('Before Expertized (moyenne ± std)')
    for ax in (ax_e, ax_b):
        ax.set_xlabel('Time')
        ax.set_ylabel('Value')
        ax.legend(title='Model')
        ax.grid(True)

    fig.suptitle(f'Métric : {metric} — Moyenne et écart‑type sur split=Val', fontsize=16)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
